<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab2_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🔧 M03 Lab 2 — Function Calling & Tool Use</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~12 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Define <b>function schemas</b> that tell the model what tools are available</li>
    <li>Parse <b>tool call responses</b> programmatically</li>
    <li>Complete the full <b>tool call loop</b>: call → execute → return</li>
    <li>Understand why function calling is the <b>foundation of AI agents</b></li>
  </ol>
</div>

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai tiktoken pydantic
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import json
from pydantic import BaseModel
from openai import OpenAI
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ What is Function Calling?</h2>
</div>

In Lab 1, we used **JSON mode** to get structured output from the model. But what if we want the model to **take actions** — like checking the weather, searching a database, or sending an email?

**Function calling** lets you define **tools** the model can invoke. Instead of returning text, the model returns a structured function call with arguments that *you* execute.

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 The Big Picture:</b>
  <ul style="margin: 6px 0 0;">
    <li><b>JSON mode</b> = structured <i>output</i> (model gives you data)</li>
    <li><b>Function calling</b> = structured <i>actions</i> (model tells you what to do)</li>
    <li>This is the <b>foundation of AI agents</b> — models that can interact with the real world</li>
  </ul>
</div>

In [ ]:
# ============================================================
# 🌤️ Step 1: Define a Tool Schema
# ============================================================
# This tells the model: "You have access to a weather function"

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"},
                    "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    }
]

# Let's see how the model responds when it has tools available
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What's the weather like in Boston today?"}],
    tools=tools,
    tool_choice="auto",  # Let the model decide whether to call a tool
)

# The model didn't return text — it returned a TOOL CALL
tool_call = response.choices[0].message.tool_calls[0]
print(f"🔧 Function: {tool_call.function.name}")
print(f"📋 Arguments: {tool_call.function.arguments}")

args = json.loads(tool_call.function.arguments)
print(f"\n✅ Parsed → city: {args['city']}, units: {args.get('units', 'not specified')}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Notice the model didn't return text — it returned a <b>function call</b> with structured arguments. The model <i>decided</i> to use the tool because the user asked about weather and a weather tool was available. This is the magic of function calling: the model <b>reasons about which tool to use</b>.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ The Tool Call Loop</h2>
</div>

The full flow has **3 steps**:

1. **User asks** → model returns a `tool_call` (not text)
2. **You execute** the function with the model's arguments
3. **You send the result back** → model generates a human-readable response

This loop is how every AI assistant (ChatGPT, Claude, etc.) handles tools like web search, code execution, and file uploads.

In [ ]:
# ============================================================
# 🔄 Step 2 & 3: Execute the Function → Send Result Back
# ============================================================

# Step 2: Execute the function (simulated — in production this would call a real API)
def get_weather(city: str, units: str = "fahrenheit") -> dict:
    """Simulated weather function."""
    weather_data = {
        "Boston": {"temp": 42, "condition": "Partly cloudy", "humidity": 65},
        "Miami": {"temp": 78, "condition": "Sunny", "humidity": 80},
        "Seattle": {"temp": 52, "condition": "Rainy", "humidity": 90},
    }
    data = weather_data.get(city, {"temp": 55, "condition": "Unknown", "humidity": 50})
    return {"city": city, "units": units, **data}

result = get_weather(**args)
print(f"📊 Function returned:")
pp(result)

# Step 3: Send the result back to the model
follow_up = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "What's the weather like in Boston today?"},
        response.choices[0].message,          # The assistant's tool call
        {
            "role": "tool",                    # New role: "tool"
            "tool_call_id": tool_call.id,      # Must match the tool call ID
            "content": json.dumps(result),     # Function result as JSON string
        }
    ],
    tools=tools,
)

# Now the model generates a natural language response!
print("\n🤖 Model's response to the user:")
show_response(follow_up)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Multiple Tools — The Model Chooses</h2>
</div>

The real power of function calling appears when you give the model **multiple tools** and let it decide which one(s) to use based on the user's request.

# ============================================================
# 🛠️ Multiple Tools: Weather + Calculator + Translator
# ============================================================

multi_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "translate",
            "description": "Translate text to another language",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to translate"},
                    "target_language": {"type": "string", "description": "Target language"}
                },
                "required": ["text", "target_language"]
            }
        }
    }
]

# Test with different requests — watch which tool the model picks!
test_requests = [
    "What's the weather in Tokyo?",
    "What is 15% tip on a $85 dinner bill?",
    "How do you say 'Thank you very much' in Japanese?",
]

for request in test_requests:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": request}],
        tools=multi_tools,
        tool_choice="auto",
    )
    tc = r.choices[0].message.tool_calls[0]
    print(f"🧑 User: \"{request}\"")
    print(f"   🔧 Tool: {tc.function.name}")
    print(f"   📋 Args: {tc.function.arguments}\n")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Real-World Example: Resume Parser</h2>
</div>

Let's combine everything from M03 — **JSON mode + Pydantic + function calling concepts** — to build a structured resume extractor. This is a common real-world use case in HR tech.

In [ ]:
# ============================================================
# 📄 Resume Parser: JSON Mode + Pydantic
# ============================================================

class Experience(BaseModel):
    company: str
    title: str
    duration: str

class ResumeData(BaseModel):
    name: str
    email: str
    skills: list[str]
    experience: list[Experience]
    education: str

resume_text = """
Jane Smith - jane.smith@email.com
Senior Data Scientist with 6 years of experience in ML and NLP.

EXPERIENCE
- DataCorp Inc. | Senior Data Scientist | 2021-Present
  Led NLP team, built production recommendation engine
- StartupAI | ML Engineer | 2019-2021
  Developed computer vision pipeline for quality inspection

SKILLS: Python, PyTorch, TensorFlow, SQL, NLP, Computer Vision, MLOps

EDUCATION: M.S. Computer Science, MIT (2019)
"""

r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Extract structured resume data. Return JSON with keys: name, email, skills (list), experience (list of {company, title, duration}), education."},
        {"role": "user", "content": resume_text}
    ],
    response_format={"type": "json_object"},
)

# Parse and validate with Pydantic
resume = ResumeData(**json.loads(r.choices[0].message.content))

print(f"👤 Name: {resume.name}")
print(f"📧 Email: {resume.email}")
print(f"🛠️ Skills: {', '.join(resume.skills)}")
print(f"\n💼 Experience:")
for exp in resume.experience:
    print(f"   - {exp.title} at {exp.company} ({exp.duration})")
print(f"\n🎓 Education: {resume.education}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "When should you use function calling over JSON mode?",
        "options": [
            "When you need the cheapest option",
            "When you want the model to decide which action to take from a set of available tools",
            "When you only need simple text responses",
            "When working with Gemini models"
        ],
        "answer": 1,
        "explanation": "Function calling is ideal when the model needs to choose between multiple tools and provide structured arguments — the foundation of AI agents."
    },
    {
        "q": "What role do you use when sending function results back to the model?",
        "options": [
            "\"system\"",
            "\"user\"",
            "\"assistant\"",
            "\"tool\""
        ],
        "answer": 3,
        "explanation": "The 'tool' role is specifically designed for sending function execution results back to the model, along with the matching tool_call_id."
    },
    {
        "q": "In the tool call loop, who actually executes the function?",
        "options": [
            "The LLM runs the code directly",
            "OpenAI's servers run it",
            "Your code executes it — the model only provides the arguments",
            "The function runs automatically via the API"
        ],
        "answer": 2,
        "explanation": "The model only decides WHAT to call and WITH WHAT arguments. Your code is responsible for actually executing the function and returning the result."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Define a Product Search Tool

Define a function schema for `search_products` with these parameters:
- `query` (string, required) — search keywords
- `category` (enum: electronics, clothing, books — required)
- `max_price` (number, optional) — price filter

In [ ]:
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've mastered function calling — the foundation of AI agents and tool-using LLMs.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Function schemas</b> tell the model what tools are available and their parameters</li>
      <li>The <b>tool call loop</b>: user → model decides tool → you execute → model summarizes</li>
      <li>With <b>multiple tools</b>, the model acts as a router — choosing the right tool for each request</li>
      <li>Combine <b>JSON mode + Pydantic + function calling</b> for production-grade data pipelines</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Assignment M03:</b> Build a structured data extractor for a domain of your choice. Define Pydantic models, use JSON mode on 3+ inputs, validate with Pydantic. Submit code + outputs + edge case observations.</p>
</div>

In [ ]:
# --- Test ---
assert tc.function.name == "search_products", f"Expected 'search_products', got '{tc.function.name}'"
args_ex1 = json.loads(tc.function.arguments)
assert "query" in args_ex1, "Missing 'query' in arguments"
assert "category" in args_ex1, "Missing 'category' in arguments"
show_expected('Function: search_products\nArguments: {"query": "wireless headphones", "category": "electronics", "max_price": 100}')

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the model correctly identify the category as "electronics"? _[Your answer]_

2. Did it extract the max_price of 100 from the natural language request? _[Your answer]_

3. What happens if you ask "Find me a good mystery novel"? Does it still pick the right category? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these experiments on your own:</p>
  <ol style="font-size: 14px;">
    <li>Add a 4th tool to the multi-tool example (e.g., <code>send_email</code> or <code>search_web</code>) — does the model route correctly?</li>
    <li>Try <code>tool_choice="required"</code> instead of <code>"auto"</code> — what changes?</li>
    <li>Ask a question that doesn't match any tool (e.g., "Tell me a joke") — what does the model return?</li>
  </ol>
</div>

In [ ]:
# --- Test ---
assert tc.function.name == "search_products", f"Expected 'search_products', got '{tc.function.name}'"
args_ex1 = json.loads(tc.function.arguments)
assert "query" in args_ex1, "Missing 'query' in arguments"
show_expected('Function: search_products\nArguments: {"query": "wireless headphones", "category": "electronics", "max_price": 100}')

---
## Summary

- **Function calling**: Define tool schemas → model returns structured calls → you execute
- **Tool call loop**: user message → assistant tool_call → tool result → assistant response
- **Real-world**: Combine JSON mode + Pydantic for robust extraction pipelines

---

## Assignment M03

Build a **structured data extractor** for a domain of your choice (menu, job posting, product review, etc.).

**Requirements:**
1. Define a Pydantic model for your domain
2. Use JSON mode to extract data from at least 3 sample inputs
3. Validate outputs with Pydantic

**Submit:** Code + 3 sample inputs/outputs + 1-paragraph observation on edge cases